# COde 00 — Research wheel bank v6 / package 0.6.0

Chạy ONLINE trên Linux/Kaggle có cùng Python/scientific stack với phiên GPU đích.
Nếu RTX PRO 6000 chặn Internet: notebook00 dùng Accelerator=None (CPU), Internet ON.
Notebook01/02 mới dùng Pro6000, Internet OFF. Lỗi đọc Secrets chưa chứng minh HF token sai.
TARGET_ENVIRONMENT có thể giữ inventory từ MODE='inspect' trên GPU để so trước khi tải.
Gắn source 0.6.0 đã giải nén. Chưa có prepared release vẫn chạy: RUN_MINI_REHEARSAL=False.
Bank chứa project wheel, dependency wheelhouse + requirements.lock, source/configs và pretrained weights có checksum.
Chấp nhận điều khoản DINOv3-B/L, thêm Kaggle Secret HF_TOKEN và cấp quyền cho notebook.
Hướng dẫn: _support/KAGGLE_RUNBOOK.md. Bank cũ cần tạo lại; không đổi prepared split/features.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
TARGET_ENVIRONMENT = None  # paste inventory from Pro6000 inspect; CPU stack must match
MODE = 'prepare'  # inspect để xem môi trường GPU đích; prepare để tải online
PROJECT_ROOT_OVERRIDE = None
BANK_ROOT = Path('/kaggle/working/clindistill_research_v6')
REUSE_MODELS_FROM = None  # prior bank5/6 root; verified weights/revisions only
RUN_MINI_REHEARSAL = False
MINI_ROOT_OVERRIDE = None

def find(predicate):
    found = set()
    for base in (Path.cwd(), Path('/kaggle/input')):
        if not base.exists(): continue
        for current, dirs, files in os.walk(base):
            p = Path(current)
            if predicate(p, dirs, files): found.add(p.resolve())
            dirs[:] = [d for d in dirs if d not in {'Images','images','.git','.cache','_support','models','wheels','wheelhouse','official_code','_archive'} and not d.lower().startswith('qwen')]
            if len(p.relative_to(base).parts) >= 8: dirs[:] = []
    return sorted(found, key=str)
def one(override, found, name):
    if override: return Path(override).resolve()
    if len(found) != 1: raise RuntimeError(f'{name}: {found}. Set *_OVERRIDE.')
    return found[0]
source = one(PROJECT_ROOT_OVERRIDE, find(lambda p,d,f:
    (p / 'clindistill/research').is_dir() and 'configs' in d and p != (BANK_ROOT / 'source').resolve()), 'Source 0.6.0')
spec = importlib.util.spec_from_file_location('research_bank', source / '_support/research_bank.py')
bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
print(json.dumps(bank.inventory(), indent=2))
# GPU runtime preflight before downloading large assets
runtime_spec = importlib.util.spec_from_file_location('research_hardware', source / 'clindistill/research/runtime.py')
runtime = importlib.util.module_from_spec(runtime_spec); runtime_spec.loader.exec_module(runtime)
print(json.dumps(runtime.hardware(), indent=2))
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    if MINI_ROOT_OVERRIDE:
        mini = Path(MINI_ROOT_OVERRIDE)
    else:
        full_release = one(None, find(lambda p,d,f: 'release.json' in f and
            not bank.read_json(p / 'release.json').get('debug_subset')), 'Full prepared COde release')
        full_images = one(None, find(lambda p,d,f: (p / 'Images/Photographs').is_dir() and
            'mini_dataset.json' not in f), 'Full COde images')
        mini = BANK_ROOT.parent / 'research_mini_v1'
        mini_spec = importlib.util.spec_from_file_location('mini_builder', source / '_support/kaggle_offline.py')
        mini_builder = importlib.util.module_from_spec(mini_spec); mini_spec.loader.exec_module(mini_builder)
        if not (mini / 'mini_dataset.json').exists():
            import uuid
            staging = mini.with_name(mini.name + '_staging_' + uuid.uuid4().hex[:8])
            mini_builder.make_mini_code(full_release, full_images, staging, per_class=2, independent_groups=True)
            staging.rename(mini)
        elif bank.read_json(mini / 'prepared/release.json')['source_release_id'] != bank.read_json(full_release / 'release.json')['release_id']:
            raise RuntimeError('Mini thuộc release khác. Đặt MINI_ROOT_OVERRIDE tới mini đúng release.')
# Research bank v6 dispatch: use the versioned helper, never patch an older bank in memory.
if bank.BANK_SCHEMA != 6:
    raise RuntimeError('Attach source 0.6.0 with bank schema6; do not combine this notebook with an old addon.')
if MODE == 'prepare':
    if TARGET_ENVIRONMENT is None and bank.inventory()['packages']['torch'].endswith('+cpu'):
        raise RuntimeError('CPU online setup needs TARGET_ENVIRONMENT copied from GPU MODE=inspect. '
                           'The bank must record the GPU target, not the CPU Torch build.')
    if TARGET_ENVIRONMENT is not None:
        bridge = bank.require_preparation_environment(TARGET_ENVIRONMENT, bank.inventory())
        if bridge:
            print('CPU online -> CUDA target: same Torch/vision releases; all other packages match. GPU activation remains strict.')
    bank.check_online()
    bank.read_hf_token()
    info = bank.prepare(source, BANK_ROOT, target_environment=TARGET_ENVIRONMENT, reuse_models_from=REUSE_MODELS_FROM)
elif MODE == 'inspect':
    print('INSPECT ONLY: no network, Secrets or downloads. Copy the inventory JSON into TARGET_ENVIRONMENT on the online CPU notebook.')
else:
    raise ValueError('MODE must be inspect or prepare.')


## Rehearsal tùy chọn sau khi có prepared release

Lần đầu chỉ tạo bank. Sau notebook01, có thể đặt RUN_MINI_REHEARSAL=True,
gắn prepared release và ảnh để thử pretrained thật trên mini độc lập theo nhóm bệnh nhân.
Đây là kiểm tra phần mềm, không phải kết quả nghiên cứu; không mở test.


In [ ]:
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    bank.installed_project_matches(info['project_files'])
    from clindistill.artifacts import read_json, write_json
    from clindistill.research.data import prepare_features
    from clindistill.research.training import run_study
    from clindistill.research.training import grouped_folds
    rows = [json.loads(line) for line in (mini / 'prepared/student_single_train.jsonl').read_text(encoding='utf-8').splitlines() if line]
    grouped_folds([r['single_label_index'] for r in rows], [r['split_group'] for r in rows], 2, 2026)
    models = {k: BANK_ROOT / v for k,v in info['models'].items()}
    cfg = read_json(BANK_ROOT / 'source/configs/code/research.json')
    cfg.update(debug=True, seeds=[2026], folds=2, epochs=3, patience=3, feature_dim=64,
        learning_rate=0.01, bootstrap_repetitions=10,
        methods=['student_no_kd','kd_visual','kd_multimodal','kd_confidence','kd_reliability'])
    cache = BANK_ROOT.parent / 'research_mini_features'
    with bank.block_network():
        prepare_features(mini / 'prepared', mini, models, cache, cfg)
        display(run_study(cache, BANK_ROOT.parent / 'research_mini_runs', cfg, require_gpu=False))
    write_json(BANK_ROOT / 'rehearsal.json', {'status':'passed','debug':True,'contains_test':False})
print('Save Version. 01/02/03 Add Input > Notebook Output > chọn output có research_bank_manifest.json.')


Bank gồm source, Linux wheels, DINO-B/L, BGE-M3, ResNet/Swin weights; không gồm COde đầy đủ.
Giữ bank, mini, cache và checkpoints riêng tư. Nếu chưa có quyền DINO, dừng ở tải model và hoàn tất quyền truy cập;
không có cơ chế thay model hoặc sinh kết quả giả. Resume prepare dùng lại model downloads đã có.